*本笔记本的数据、概念和初始实现由 timm 的创建者 Ross Wightman 在 Colab 中完成。我（Jeremy Howard）对分析进行了一些重构、整理和扩展，并添加了文字说明。*

## timm

[PyTorch Image Models](https://timm.fast.ai/)（timm）是由 Ross Wightman 开发的一个出色的库，提供了最先进的预训练计算机视觉模型。它类似于 Hugging Face Transformers，但专注于计算机视觉而非 NLP（而且并不局限于基于 Transformer 的模型）！

Ross 非常热心地帮助我了解如何充分利用这个库，并为我指出了其中的顶尖模型。我将在这里分享我从他那里学到的内容，以及一些额外的心得体会。

## 数据

Ross 会定期对加入 timm 的新模型进行基准测试，并将结果保存为项目 GitHub 仓库中的 CSV 文件。为了分析这些数据，我们首先来克隆该仓库：

In [ ]:
! git clone --depth 1 https://github.com/rwightman/pytorch-image-models.git
%cd pytorch-image-models/results

使用 Pandas，我们可以读取所需的两个 CSV 文件，并将它们合并在一起。

In [ ]:
import pandas as pd
df_results = pd.read_csv('results-imagenet.csv')

我们还将添加一个"family"（系列）列，用于将架构按相似特性分类归组：

Ross 告诉了我哪些模型在实际使用中效果最好，因此我会将图表限制为只展示这些模型。（我也加入了 VGG，不是因为它性能出色，而是作为对比，以展示过去几年来技术的进步有多大。）

In [ ]:
def get_data(part, col):
    df = pd.read_csv(f'benchmark-{part}-amp-nhwc-pt111-cu113-rtx3090.csv').merge(df_results, on='model')
    df['secs'] = 1. / df[col]
    df['family'] = df.model.str.extract('^([a-z]+?(?:v2)?)(?:\d|_|$)')
    df = df[~df.model.str.endswith('gn')]
    df.loc[df.model.str.contains('in22'),'family'] = df.loc[df.model.str.contains('in22'),'family'] + '_in22'
    df.loc[df.model.str.contains('resnet.*d'),'family'] = df.loc[df.model.str.contains('resnet.*d'),'family'] + 'd'
    return df[df.family.str.contains('^re[sg]netd?|beit|convnext|levit|efficient|vit|vgg')]

In [ ]:
df = get_data('infer', 'infer_samples_per_sec')

## 推理结果

以下是推理性能的测试结果（训练性能请参见最后一节）。图表说明：

- x 轴表示处理一张图片所需的秒数（<strong>注意</strong>：采用对数刻度）
- y 轴表示在 ImageNet 上的准确率
- 每个气泡的大小与测试时所用图片的尺寸成正比
- 颜色表示该架构所属的"系列"

将鼠标悬停在标记上可查看模型详情。在图例中双击可只显示某一系列；单击可显示或隐藏某一系列。

<strong>注意</strong>：在我的屏幕上，Kaggle 会截断系列选择器以及部分 Plotly 功能——若要查看完整内容，请点击"*Contents*"右侧的小箭头，将右侧目录折叠起来。

In [ ]:
import plotly.express as px
w,h = 1000,800

def show_all(df, title, size):
    return px.scatter(df, width=w, height=h, size=df[size]**2, title=title,
        x='secs',  y='top1', log_x=True, color='family', hover_name='model', hover_data=[size])

In [ ]:
show_all(df, 'Inference', 'infer_img_size')

这些模型系列的数量可能有些令人不知所措，所以我只挑选一个子集，代表我们图表中表现最佳的每个系列中的一个关键模型。我还将 convnext 模型分为那些在更大的 22,000 类别 imagenet 样本上预训练过的（`convnext_in22`）和没有预训练过的（`convnext`）。（请注意，许多性能最佳的模型都是在更大的样本上训练的——在对这些架构的整体有效性得出结论之前，请参阅相关论文了解详情。）

In [ ]:
subs = 'levit|resnetd?|regnetx|vgg|convnext.*|efficientnetv2|beit'

在这张图表中，我将通过每个家族的点添加连线，以帮助观察它们之间的比较——但请注意，我们可以看到线性拟合在这里实际上并不理想！它只是用来帮助直观地看清各组。

In [ ]:
def show_subs(df, title, size):
    df_subs = df[df.family.str.fullmatch(subs)]
    return px.scatter(df_subs, width=w, height=h, size=df_subs[size]**2, title=title,
        trendline="ols", trendline_options={'log_x':True},
        x='secs',  y='top1', log_x=True, color='family', hover_name='model', hover_data=[size])

In [ ]:
show_subs(df, 'Inference', 'infer_img_size')

从中我们可以看出，*levit* 系列模型在图像识别方面速度极快，在速度较快的模型中准确率也明显领先。这并不令人意外，因为这些模型融合了 CNN 与 Transformer 的精华，兼得两者之长。事实上，在中等速度的模型中也能看到类似的规律——表现最佳的是 ConvNeXt，它是一个纯 CNN 模型，但借鉴了 Transformer 领域的诸多思想。

在速度最慢的模型中，*beit* 的准确率最高——不过在解读这一结果时需要保持谨慎，因为它是在更大的数据集（ImageNet-21k，*vit* 模型同样使用该数据集）上训练的。

这里我再补充一张图，展示速度与参数量之间的关系。在学术论文中，参数量常被用作速度的替代指标。然而，正如我们所看到的，在相同参数量下，模型速度的差异相当大，因此参数量实际上并不是一个可靠的替代指标。

（参数量或许在一定程度上能反映模型所需的内存大小，但即便如此，它也并非总是一个理想的参考指标。）

In [ ]:
px.scatter(df, width=w, height=h,
    x='param_count_x',  y='secs', log_x=True, log_y=True, color='infer_img_size',
    hover_name='model', hover_data=['infer_samples_per_sec', 'family']
)

## 训练结果

我们现在将对训练性能重复上述分析。首先，我们获取数据：

In [ ]:
tdf = get_data('train', 'train_samples_per_sec')

现在我们可以重复上面做的同样的<em>族</em>图：

In [ ]:
show_all(tdf, 'Training', 'train_img_size')

……我们还将查看我们所选择的模型子集：

In [ ]:
show_subs(tdf, 'Training', 'train_img_size')

最后，我们应该记住，速度取决于硬件。如果您使用的不是现代 NVIDIA GPU，您的结果可能会有所不同。特别是，我怀疑基于 transformers 的模型在 CPU 上的整体性能可能会更差（尽管我需要进一步研究才能确定）。

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文档已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。虽然我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言的原文档应视为权威来源。对于重要信息，建议使用专业人工翻译。对于因使用本翻译而产生的任何误解或误释，我们概不负责。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
